In [ ]:
# V5.1 CANARY — operator launcher (Rung B, development GPU)
# Frozen executable: CANARY_EXECUTABLE_COMMIT (see constant below).
# The notebook checks out THAT commit — never a moving branch — so notebook
# edits can never silently change scientific execution.
CANARY_EXECUTABLE_COMMIT = "6fb1bef4961709584e5e20bc7b6833eb36205ecf"
BRANCH = "cymek-v51-canary"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
DRIVE_ROOT = "/content/drive/MyDrive/CYMEK/V5_1_CANARY"
BLOB_HASHES = {  # critical blobs verified after checkout (sha256)
    # filled by the freeze tooling at launcher-commit time
}
import sys
assert sys.version_info[:2] >= (3, 10)
print("launcher identity:", CANARY_EXECUTABLE_COMMIT)

In [ ]:
# 1) CUDA/T4 required
import torch
assert torch.cuda.is_available(), "FAIL_CLOSED: this launcher requires a CUDA GPU (T4)"
name = torch.cuda.get_device_name(0)
assert "T4" in name, f"WARN/FAIL_CLOSED: expected T4, got {name}"
props = torch.cuda.get_device_properties(0)
print(name, f"{props.total_memory/2**30:.1f} GiB")

In [ ]:
# 2) clone + checkout EXACT commit (fail if the object is absent)
import subprocess, pathlib
REPO = pathlib.Path("/content/An-Ra-the-new-AGI")
if not REPO.exists():
    subprocess.run(["git","clone",REPO_URL,str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"fetch","origin",CANARY_EXECUTABLE_COMMIT,"--depth","1"],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","-q","--detach",CANARY_EXECUTABLE_COMMIT],check=True)
head = subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
assert head == CANARY_EXECUTABLE_COMMIT, (head, CANARY_EXECUTABLE_COMMIT)
print("checked out:", head)

In [ ]:
# 3) verify critical blob hashes
import hashlib, json
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
blobs = {
 "anra_v5/v51_canary_run.py": None,
 "anra_v5/v51_canary_data.py": None,
 "experiments/V5_1_CANARY/PREREGISTRATION.json": None,
 "v5_training/step.py": None,
 "v5_training/schedule.py": None,
 "v5_training/optimizer.py": None,
 "v5_training/checkpoint.py": None,
}
for rel in blobs:
    h = sha(REPO/rel)
    if BLOB_HASHES and rel in BLOB_HASHES:
        assert h == BLOB_HASHES[rel], f"FAIL_CLOSED: blob hash drift {rel}"
    print(f"{h[:16]}  {rel}")
print("blob verification complete")

In [ ]:
# 4) mount Drive (BEFORE any persistent-state scan)
from google.colab import drive
drive.mount("/content/drive")
import pathlib
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print("drive root:", DRIVE_ROOT)

In [ ]:
# 5) preflight: one real update through the production backend on CUDA
!cd {REPO} && python -m anra_v5.v51_canary_run --mode preflight --rung B --cuda --bfloat16

In [ ]:
# 6) scan persistent state -> START / RESUME / COMPLETE / FAIL_CLOSED
import os
os.environ["V51_CANARY_STATE_ROOT"] = DRIVE_ROOT
!cd {REPO} && python -m anra_v5.v51_canary_run --mode scan --state-root {DRIVE_ROOT}/state

In [ ]:
# 7) run / resume (idempotent: scan decides; the schedule never rewarm)
#    TOTAL target updates for Rung B: 120 (491,520-token WSD budget)
!cd {REPO} && python -m anra_v5.v51_canary_run --mode run --rung B --cuda --bfloat16 --updates 120 --checkpoint-every 16 --state-root {DRIVE_ROOT}/state

In [ ]:
# 8) evaluate on development (iteration only)
!cd {REPO} && python -m anra_v5.v51_canary_run --mode evaluate --rung B --cuda --state-root {DRIVE_ROOT}/state

In [ ]:
# 9) finalize: SEALED consumed exactly once + pass gates + verdict
!cd {REPO} && python -m anra_v5.v51_canary_run --mode finalize --rung B --cuda --state-root {DRIVE_ROOT}/state

In [ ]:
# 10) package final/partial bundle with manifests + SHA256
import subprocess, os
state = os.path.join(DRIVE_ROOT, "state")
out = DRIVE_ROOT
res = subprocess.run(["bash","-lc",
  f"cd {REPO} && zip -r {out}/CYMEK_V51_CANARY_RESULTS.zip experiments/V5_1_CANARY/receipts docs/cymek/v51_canary -x '*.pyc'"],
  capture_output=True, text=True)
print(res.stdout[-2000:], res.stderr[-2000:])
import hashlib, pathlib
for z in pathlib.Path(out).glob("CYMEK_V51_CANARY_*.zip"):
    print(z.name, hashlib.sha256(z.read_bytes()).hexdigest())